# CropCop Track B — 00 Readiness & Materialization

**Purpose:** one-time construction of the two immutable Track-B v4 Kaggle input bundles. This notebook performs **no protected external R07 predictions**.

### Kaggle setup
- Account: `ranamuhammadahmed6`
- Accelerator: **T4 x2**
- Internet: **ON**
- Secret: **KAGGLE_API_TOKEN**
- Attach exactly these five source datasets:
  1. `ranamuhammadahmed6/cropcop-finalized-v8-11-2026-1`
  2. `sabahatabbas/sec-je-r07-cnxtt-context-s1-8904b100d223-a01`
  3. `sabahatabbas/cropcop-r07-cnxtt-context-s2-abce1197-56023042`
  4. `sabahatabbas/cropcop-r07-cnxtt-context-s3-f13ca687-56023042`
  5. `ranamuhammadahmed6/cropcop-secondary-g1-8904b100`

Do not attach GVLiD or Irish Potato manually. This readiness notebook acquires those exact published versions once, validates them, packages them, publishes two private Kaggle datasets, and round-trip verifies both.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

WORK = Path('/kaggle/working')
REPO = WORK / 'ResearchWork-CropCop-trackb-v4'
REPO_URL = 'https://github.com/rana-m-ahmed/ResearchWork-CropCop.git'
SOURCE_COMMIT = '63d22dbe00a183712e086d0e5c9242f37f1dfd4e'

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git','clone','--filter=blob:none','--no-checkout',REPO_URL,str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_COMMIT], check=True)
head = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
if head != SOURCE_COMMIT:
    raise RuntimeError(f'Frozen v4 source mismatch: expected {SOURCE_COMMIT}, got {head}')

from kaggle_secrets import UserSecretsClient
token = (UserSecretsClient().get_secret('KAGGLE_API_TOKEN') or '').strip()
if not token or any(ch.isspace() for ch in token):
    raise RuntimeError('KAGGLE_API_TOKEN is missing/invalid')
os.environ['KAGGLE_API_TOKEN'] = token
del token
print('Frozen Track-B v4 source:', head)
print('Kaggle publication credential: PASS')


In [ ]:
BOOTSTRAP = REPO / 'journal_extension/scripts/bootstrap_trackb_runtime.py'
LOCKFILE = REPO / 'journal_extension/track_b_r07/requirements-trackb.lock.txt'
RECEIPT = WORK / 'TRACKB_V4_READINESS_RUNTIME.json'

subprocess.run([
    sys.executable, str(BOOTSTRAP),
    '--requirements', str(LOCKFILE),
    '--receipt', str(RECEIPT),
], cwd=REPO, check=True, env=os.environ.copy())

runtime = json.loads(RECEIPT.read_text())
if runtime.get('status') != 'PASS':
    raise RuntimeError('Track-B runtime bootstrap failed')
print(json.dumps({
    'runtime_status': runtime['status'],
    'cuda_available': runtime['probe']['cuda_available'],
    'cuda_devices': runtime['probe']['cuda_devices'],
    'versions': runtime['probe']['versions'],
}, indent=2, sort_keys=True))


In [ ]:
MATERIALIZER = REPO / 'journal_extension/scripts/trackb_v4_materialize.py'
OUT = Path('/kaggle/tmp/trackb_v4_materialization')
if OUT.exists():
    shutil.rmtree(OUT)

subprocess.run([
    sys.executable, str(MATERIALIZER),
    '--repo-root', str(REPO),
    '--input-root', '/kaggle/input',
    '--output-root', str(OUT),
    '--device', 'cuda:0',
    '--kaggle-owner', 'AUTO',
], cwd=REPO, check=True, env=os.environ.copy())


In [ ]:
receipt_path = Path('/kaggle/tmp/trackb_v4_materialization') / 'TRACKB_READINESS_RECEIPT.json'
receipt = json.loads(receipt_path.read_text())
if receipt.get('status') != 'PASS_TRACKB_INPUT_MATERIALIZATION':
    raise RuntimeError(f"Readiness did not PASS: {receipt.get('status')}")
if receipt.get('protected_external_prediction_count') != 0:
    raise RuntimeError('Readiness produced protected external predictions')
if receipt.get('v1_test_accessed') is not False:
    raise RuntimeError('Readiness reports V1-test access')

durable_receipt = Path('/kaggle/working/TRACKB_READINESS_RECEIPT.json')
shutil.copy2(receipt_path, durable_receipt)

print(json.dumps({
    'status': receipt['status'],
    'materialization_id': receipt['materialization']['materialization_id'],
    'repository_source_sha': receipt['materialization']['repository_source_sha'],
    'source_qualification_sha256': receipt['source_qualification_sha256'],
    'infrastructure_dataset': receipt['publication']['infrastructure']['slug'],
    'external_dataset': receipt['publication']['external']['slug'],
    'protected_external_prediction_count': receipt['protected_external_prediction_count'],
    'v1_test_accessed': receipt['v1_test_accessed'],
    'durable_receipt': str(durable_receipt),
    'next_step': 'Attach the exact published versions of these two datasets to TrackB_01_Final_Execution.ipynb.',
}, indent=2, sort_keys=True))
